In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_D.xpt"
csv_file = DATA_DIR / "HIQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 HIQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if len(df) != 10348:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 17:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "HIQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HIQ_D structure = "
    "10,348 rows x 17 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: HIQ_D unexpectedly contains "
        "XPORT tiny-value artifacts. "
        "Do NOT automatically convert them to zero."
    )


print(
    "PASS: HIQ_D contains no tiny-value artifacts."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected zero value(s) found. "
        "Review against the CDC codebook before proceeding."
    )


print(
    "PASS: Source contains no true numeric zeros."
)


# ============================================================
# 6. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fractional_failures.append(
            col
        )


if fractional_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fractional_failures)
    )


print(
    "PASS: All nonmissing HIQ_D values "
    "are exact whole-number identifiers/codes."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31127:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC EXPECTED FREQUENCIES
# ============================================================

EXPECTED_FREQUENCIES = {

    "HIQ011": {
        "counts": {
            1: 8400,
            2: 1917,
            7: 5,
            9: 26
        },
        "missing": 0
    },

    "HIQ031A": {
        "counts": {
            14: 5118,
            77: 1,
            99: 25
        },
        "missing": 5204
    },

    "HIQ031B": {
        "counts": {
            15: 1193
        },
        "missing": 9155
    },

    "HIQ031C": {
        "counts": {
            16: 29
        },
        "missing": 10319
    },

    "HIQ031D": {
        "counts": {
            17: 1649
        },
        "missing": 8699
    },

    "HIQ031E": {
        "counts": {
            18: 110
        },
        "missing": 10238
    },

    "HIQ031F": {
        "counts": {
            19: 229
        },
        "missing": 10119
    },

    "HIQ031G": {
        "counts": {
            20: 15
        },
        "missing": 10333
    },

    "HIQ031H": {
        "counts": {
            21: 621
        },
        "missing": 9727
    },

    "HIQ031I": {
        "counts": {
            22: 343
        },
        "missing": 10005
    },

    "HIQ031J": {
        "counts": {
            23: 261
        },
        "missing": 10087
    },

    "HIQ031AA": {
        "counts": {
            40: 5
        },
        "missing": 10343
    },

    "HIQ260": {
        "counts": {
            1: 120,
            2: 61,
            7: 4,
            9: 4
        },
        "missing": 10159
    },

    "HIQ105": {
        "counts": {
            1: 806,
            2: 80
        },
        "missing": 9462
    },

    "HIQ270": {
        "counts": {
            1: 7781,
            2: 504,
            9: 123
        },
        "missing": 1940
    },

    "HIQ210": {
        "counts": {
            1: 718,
            2: 7675,
            9: 12
        },
        "missing": 1943
    }
}


# ============================================================
# 9. VALIDATE ALL CDC FREQUENCIES
# ============================================================

print("\n--- CDC FREQUENCY CHECKS ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<9} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: All HIQ_D frequencies match CDC."
)


# ============================================================
# 10. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# HIQ_D has:
#
# - no tiny artifacts
# - no zero restorations
# - no genuine decimal measurements
# - no recalculation
#
# Therefore expected values are exactly the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 11. CLEAN SEQN + QUESTIONNAIRE CODE FORMATTING
# ============================================================

#
# All values were proven to be whole-number
# identifiers/questionnaire codes.
#
# Examples:
#
# SEQN:
# 31127.0 -> 31127
#
# HIQ011:
# 1.0 -> 1
#
# HIQ031A:
# 14.0 -> 14
#
# HIQ031AA:
# 40.0 -> 40
#
# Missing:
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and questionnaire fields "
    "converted to clean integer formatting."
)


# ============================================================
# 12. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: No HIQ_D numeric values changed."
)


# ============================================================
# 13. EXPORT CSV
# ============================================================

#
# No:
#
# - float_format
# - rounding
# - recalculation
#
# All fields are Int64 questionnaire fields.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 14. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 15. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 17 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 17. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 18. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 10348:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 19. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# All HIQ_D fields are whole-number identifiers/codes.
#
# Expected:
#
# ""                if missing
# str(int(value))   otherwise
#
# Examples:
#
# 31127.0 -> "31127"
# 1.0     -> "1"
# 14.0    -> "14"
# 99.0    -> "99"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "HIQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number questionnaire representation."
)


# ============================================================
# 20. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 23. FINAL CDC FREQUENCY CHECK AFTER EXPORT
# ============================================================

print("\n--- FINAL CSV FREQUENCY CHECK ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        csv_df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CSV frequencies changed."
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: CSV missing count changed."
        )


print(
    "PASS: Final CSV frequencies still match CDC."
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HIQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny artifacts in XPT:",
    0
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    0
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HIQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 HIQ_D XPT -> CSV
Rows: 10,348
Columns: 17
PASS: HIQ_D structure = 10,348 rows x 17 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: HIQ_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Source contains no true numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
HIQ011    : fractional values = 0
HIQ031A   : fractional values = 0
HIQ031B   : fractional values = 0
HIQ031C   : fractional values = 0
HIQ031D   : fractional values = 0
HIQ031E   : fractional values = 0
HIQ031F   : fractional values = 0
HIQ031G   : fractional values = 0
HIQ031H   : fractional values = 0
HIQ031I   : fractional values = 0
HIQ031J   : fractional values = 0
HIQ031AA  : fractional values = 0
HIQ260    : fractional values = 0
HIQ105    : fractional values = 0
HIQ270    : fractional values = 0
HIQ210    : fractional values = 0
PASS: All nonmissing HIQ_D values are exact whole-number identifiers/codes.

--- SEQN CHECK ---
Missing: 0
Dupli